In [ ]:
from hglm.experiment import ExperimentImageOnly, get_manova, get_wilks, wilks_to_chi2
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from tqdm import tqdm

sns.set()

def get_chi2(exp_iter):
    """ computes chi2 stat of each experiment (df assumed constant) """
    chi2_observed = []    
    for exp in tqdm(exp_iter):
        e, h = get_manova(exp.x, exp.y, contrast=exp.contrast) 
        wilks = get_wilks(e, h)
        chi2, df = wilks_to_chi2(wilks, contrast=contrast, b=b, n=num_img * np.prod(shape))
        chi2_observed.append(chi2)
    
    return chi2_observed, df

def sample_plot_chi2(exp_iter, **kwargs):
    """ observes & predicts chi2 distribution over iterator of Experiment """
    # observed
    chi2_observed, df = get_chi2(exp_iter, **kwargs)
    
    # predicted
    x = np.linspace(0, max(chi2_observed), 100)
    pdf = stats.chi2.pdf(x, df)
    
    # fitted
    _df, _loc, _scale = stats.chi2.fit(chi2_observed)
    pdf_fit = stats.chi2.pdf(x, _df, loc=_loc, scale=_scale)

    # plot
    plt.plot(x, pdf, 'r-', label=f'Theoretical Chi-Sq(df={int(df)})')
    plt.plot(x, pdf_fit, 'k', label=f'Fitted Chi-Sq(df={_df})', linestyle=':')
    plt.hist(chi2_observed, bins=100, density=True, alpha=0.6, color='g', label='Observed Chi2')
    plt.xlabel('Chi2')
    plt.legend()

    plt.tight_layout()
    plt.show()

# wilks is chi2 distributed (bartlett's approximation)

sanity check: under the null hypothesis wilks lambda is approximately chi square distributed

In [ ]:
def normal_exp_iter(n, contrast, **kwargs):
    for seed in range(n):
        exp = ExperimentImageOnly.from_gauss(**kwargs)
        exp = exp.sample_x(seed=seed, add_bias=True, contrast=contrast)
        
        yield exp

n = 1000
num_img = 10
shape=(2, 3, 4)
b = 2
contrast=np.array([True, True, True])

exp_iter = normal_exp_iter(n=n, contrast=contrast, num_img=num_img, shape=shape, b=b)
sample_plot_chi2(exp_iter)

# sanity check: does real data follow chi2?

- under typical assumptions ... no

In [ ]:
# human connectome project data
folder = '/home/matt/Dropbox/pnl_hglm/data/hcp100_lowres/image'
exp_hcp = ExperimentImageOnly.from_search(folder=folder,
                                          sbj_regex=r'[\d]{6}',
                                          img_glob_dict={'FA': '*_FA.nii.gz',
                                                         'MD': '*_MD.nii.gz'})
exp_hcp = exp_hcp.sample_x(a=2, seed=1, add_bias=True)
    

In [ ]:
from hglm.effect import ExtenterSphere, ExtenterMinVar
from hglm.experiment import ExperimentScaled

def hcp_exp_iter(exp_hcp, n, extenter):

    for seed in range(n):
        # trim experiment to reasonable size (for speedup)
        mask = extenter(y=exp_hcp.y, mask_idx=exp_hcp.mask_idx, seed=seed, contiguous=True)
        _exp = exp_hcp.apply_mask(mask)

        # scale normalize before sampling minimum variance (each feature given
        # equal weight in sampling extent)
        _exp = ExperimentScaled.from_exp(_exp)

        yield _exp

extenter = ExtenterMinVar(n=50)
exp_iter = hcp_exp_iter(exp_hcp, n=100, extenter=extenter)
sample_plot_chi2(exp_iter)

# sanity check: does real data follow chi2?

- if we sample one location (minimum var) and then permute?

In [ ]:
def hcp_exp_iter(exp_hcp, n, num_voxel, seed=0):
    # sample region extent
    extenter = ExtenterMinVar(n=num_voxel)
    mask = extenter(y=exp_hcp.y, mask_idx=exp_hcp.mask_idx, seed=seed, contiguous=True)
    exp = exp_hcp.apply_mask(mask)
    
    # scale normalize
    exp = ExperimentScaled.from_exp(exp)

    for _seed in range(n):
        yield exp.permute(_seed)
        
exp_iter = hcp_exp_iter(exp_hcp, n=100, num_voxel=250, seed=3)
sample_plot_chi2(exp_iter)

# new approach: estimate null distribution as normal from permutations
(use - log transform on wilk's lambda)

In [ ]:
def get_wilks_iter(exp_iter):
    """ computes wilks of each experiment """
    wilks_observed = []    
    for exp in tqdm(exp_iter):
        e, h = get_manova(exp.x, exp.y, contrast=exp.contrast) 
        wilks_observed.append(get_wilks(e, h))
    
    return wilks_observed

def sample_plot_wilks(exp_iter, **kwargs):
    """ observes & predicts chi2 distribution over iterator of Experiment """
    # observed
    wilks_observed = get_wilks_iter(exp_iter, **kwargs)
    
    
    
    fig, ax = plt.subplots(1, 2)
    for _ax, x, label in zip(ax, 
                             (wilks_observed, -np.log(wilks_observed)),
                            ('wilks', '- log wilks')):
        plt.sca(_ax)
    
        # fitted
        loc, scale = stats.norm.fit(x)
        _, pval = stats.shapiro(x)
        
        _x = np.linspace(min(x), max(x), 100)
        pdf = stats.norm.pdf(_x, loc=loc, scale=scale)

        # plot
        plt.plot(_x, pdf, 'r-')
        plt.hist(x, bins=30, density=True, alpha=0.6, color='g')
        plt.xlabel(label)
        _ax.set_title(f'Fitted Normal (shapiro pval={pval:.3f})')

    fig.set_size_inches(10, 5)
    plt.tight_layout()
    plt.show()

In [ ]:
exp_iter = hcp_exp_iter(exp_hcp, n=1000, num_voxel=100, seed=3)
sample_plot_wilks(exp_iter)